# get some quality filtering based on trainset perf


###  Get data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import hydra
from hydra.utils import get_original_cwd
import os
from omegaconf import DictConfig, OmegaConf
from dataclasses import dataclass
from typing import List, Dict, Any

from IPython.display import display



In [ ]:
# Load config
import sys
import os
from pathlib import Path


# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

# Import Hydra config utilities
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize

# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"config")
print(f"Config path: {config_path}")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")
print(OmegaConf.to_yaml(cfg))



In [ ]:
from simulations.loop_components.dj_wrappers import DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper
# from simulations.loop_components.recording_file_copier import copy_rec_files,create_directory_structure
# from simulations.loop_components.stimulus import create_rf_avi_from_roi_ids, create_full_avi_from_roi_id_and_seed
# from simulations.loop_components.utils import log

In [ ]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore
                data_subfolders=cfg.data_subfolders, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore

                    )



In [ ]:

# Load config and tables
dj_table_holder.load_config()
dj_table_holder.load_tables()
# print(" loaded and configured successfully")
# dj_table_holder.clear_tables("all")

# dj_table_holder.setup()

In [ ]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)


In [ ]:
# dj_table_holder.clear_tables("experiment")
# preprocessor.upload_iteration_metadata()

In [ ]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
field_key = dj_table_holder("Field")().proj().fetch1()
print(field_key)

In [ ]:
from simulations.gui.integrated_autorois import InteractiveRoiCanvas
# import ipywidgets as widgets
# from ipycanvas import MultiCanvas
from IPython.display import display     


In [38]:
online_analysis_gui = InteractiveRoiCanvas(
    dj_table_holder=dj_table_holder,
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper],
    field_key=field_key,
    canvas_width=30,
    )

Load model weights for cpu from checkpoint /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/dropout_and_aug_regul.ckpt using config /gpfs01/euler/data/Resources/AutoROIs/models/UNET_v0.1.0/sd_images.yaml


In [39]:
display(online_analysis_gui.start_gui())

In [40]:
quality_type_analysis_wrapper.compute_analysis()

CelltypeAssignment: 100%|██████████| 1/1 [00:06<00:00,  6.80s/it]


# Look at metrics

In [47]:
dj_table_holder.multiprocessing_threads = 1
dj_table_holder("OpenRetinaHoeflingFormat").delete()

[2025-08-19 18:04:55,292][INFO]: Deleting 1 rows from `ageuler_ssuhai_closed_loop`.`open_retina_hoefling_format`
--- Logging error ---
Traceback (most recent call last):
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
ValueError: I/O operation on closed file.
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/tornado/platform/asy

1

In [31]:
random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    model_configs=cfg.model_configs, # type: ignore
    seeds = [111]
)

In [53]:
from openretina.data_io.base import compute_data_info
from openretina.models.core_readout import load_core_readout_model
import torch.utils.data as data
import torch
import lightning.pytorch
from openretina.data_io.cyclers import LongCycler, ShortCycler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [55]:

### Set cache folder
os.environ["OPENRETINA_CACHE_DIRECTORY"] = cfg.model_configs.paths.cache_dir

### Display log directory for ease of access


dataloaders = hydra.utils.instantiate( # dict[str, dict[str, DataLoader]]
    cfg.model_configs.dataloader,
    neuron_data_dictionary=random_seed_mei_wrapper.neuron_data_dict,
    movies_dictionary=random_seed_mei_wrapper.movies_dict,
)

data_info = compute_data_info(random_seed_mei_wrapper.neuron_data_dict, random_seed_mei_wrapper.movies_dict)

train_loader = data.DataLoader(
    LongCycler(dataloaders["train"], shuffle=True), batch_size=None, num_workers=0, pin_memory=True
)
valid_loader = ShortCycler(dataloaders["validation"])



# ## Assign missing n_neurons_dict to model
# cfg.model.n_neurons_dict = data_info["n_neurons_dict"]
# log.info(f"Instantiating model <{cfg.model._target_}>")
# model = hydra.utils.instantiate(cfg.model, data_info=data_info)

## Model init
load_model_path = cfg.model_configs.paths.get("load_model_path")

is_gru_model = "gru" in cfg.model_configs.model._target_.lower() if hasattr(cfg.model_configs.model, "_target_") else False
model = load_core_readout_model(load_model_path, DEVICE, is_gru_model=is_gru_model)


# add new readouts and modify stored data in model
model.readout.add_sessions(data_info["n_neurons_dict"])  # type: ignore
model.update_model_data_info(data_info)

if cfg.get("only_train_readout") is True:
    model.core.requires_grad_(False)

### Logging

### Callbacks
callbacks = [
    hydra.utils.instantiate(callback_params) for callback_params in cfg.get("training_callbacks", {}).values()
]

### Trainer init
trainer: lightning.Trainer = hydra.utils.instantiate(cfg.model_configs.trainer, callbacks=callbacks)
trainer.fit(model=model, train_dataloaders=train_loader, val_dataloaders=valid_loader)

# ### Testing
# short_cyclers = [(n, ShortCycler(dl)) for n, dl in dataloaders.items()]
# dataloader_mapping = {f"DataLoader {i}": x[0] for i, x in enumerate(short_cyclers)}
# test_results = trainer.test(model, dataloaders=[c for _, c in short_cyclers], ckpt_path="best")

Creating movie dataloaders:   0%|          | 0/1 [00:00<?, ?it/s]

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                        | Params | Mode 
-------------------------------------------------------------------------
0 | core             | SimpleCoreWrapper           | 12.4 K | train
1 | readout          | MultiGaussianReadoutWrapper | 65.7 K | train
2 | loss             | PoissonLoss3d               | 0      | train
3 | correlation_loss | CorrelationLoss3d           | 0      | train
-------------------------------------------------------------------------
78.1 K    Trainable params
0         Non-trainable params
78.1 K    Total params
0.312     Total estimated model params size (MB)
84        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/gpfs01/euler/User/ssuhai/.local/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.
/gpfs01/euler/User/ssuhai/.local/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.
/gpfs01/euler/User/ssuhai/.local/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (2) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


In [59]:
all_targets.shape

(1, 750, 66)

In [ ]:

import numpy as np

def get_single_neuron_test_correlations(dataloaders, 
                                        model):
    neuron_correlations = {}


    for session_id, session_dataloader in dataloaders["test"].items():
        all_preds, all_targets = [], []
        
        # Run model on all test batches
        with torch.no_grad():
            model.eval()
            model.to(DEVICE)
            for batch in session_dataloader:
                inputs, targets = batch
                inputs = inputs.to(DEVICE)
                predictions = model(inputs, data_key=session_id)
                all_preds.append(predictions.cpu())
                all_targets.append(targets.cpu())
        
        # Concatenate batch results and compute correlations
        all_preds = torch.cat(all_preds, dim=0).numpy()
        all_targets = torch.cat(all_targets, dim=0).numpy()
        
        # Fast vectorized correlation calculation
        session_correlations = {}
        num_neurons = all_targets.shape[-1]

        conv_eats_n_frames = all_targets.shape[1] - all_preds.shape[1]
        for i in range(num_neurons):
            pred = all_preds[..., i].flatten()
            target = all_targets[:,conv_eats_n_frames:, i].flatten()
            corr = np.corrcoef(pred, target)[0, 1] if np.var(pred) > 0 and np.var(target) > 0 else 0
            session_correlations[i] = corr
            
        neuron_correlations[session_id] = session_correlations
        

In [63]:
np.mean(list(neuron_correlations["online_session_1_ventral1_20250717"].values()))

0.5227675720416352

In [48]:
random_seed_mei_wrapper.compute_analysis()

Original dataset contains 103 neurons over 1 fields
 ------------------------------------ 
Dropped 0 fields that did not contain the target cell types (1 remaining)
Overall, dropped 14 neurons of non-target cell types (-13.59%).
 ------------------------------------ 
Dropped 0 fields with quality indices below threshold (1 remaining)
Overall, dropped 13 neurons over quality checks (-14.61%).
 ------------------------------------ 
Dropped 0 fields with classifier confidences below 0.25
Overall, dropped 10 neurons with classifier confidences below 0.25 (-13.16%).
 ------------------------------------ 
 ------------------------------------ 
Final dataset contains 66 neurons over 1 fields
Total number of cells dropped: 37 (-35.92%)


Upsampling natural spikes traces to get final responses.:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
dj_table_holder("CascadeTraces")() 

In [ ]:

(dj_table_holder("PreprocessTraces")() & dict(stim_name="mouse_cam")).proj()

In [ ]:
dj_table_holder("OpenRetinaHoeflingFormat")().delete()

In [ ]:
random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    model_configs=cfg.model_configs, 
    seeds= [123,456,789]) 

In [ ]:
dj_table_holder.multiprocessing_threads = 1

In [ ]:

## model training 
session_dict_raw = dj_table_holder('OpenRetinaHoeflingFormat')().extract_data()

# preprocess and filter further 
movies_dict = load_stimuli(model_configs)

neuron_data_dict = preprocess_for_openretina(self.session_dict_raw,self.model_configs)


# load and refine model
self.model = train_model_online(self.model_configs,self.neuron_data_dict,movies_dict)


In [ ]:
from openretina.utils.h5_handling import load_h5_into_dict
from openretina.data_io.hoefling_2024.responses import filter_responses, make_final_responses


In [ ]:
# load OR data
responses_path = "/gpfs01/euler/User/ssuhai/openretina_cache/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.h5"

responses_dict = load_h5_into_dict(file_path=responses_path)

filtered_responses_dict = filter_responses(responses_dict, **cfg.model_configs.quality_checks)

final_responses = make_final_responses(filtered_responses_dict, response_type="natural")

In [ ]:
random_seed_mei_wrapper 